In [6]:
!pip install -q -U ultralytics pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 4.1 MB/s eta 0:00:00


In [7]:
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Kaggle GPU is not enabled.")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics: 8.4.128
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level <= 4:
        print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/muki2003
/kaggle/input/datasets/muki2003/yolo-drone-detection-dataset
/kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset


In [13]:
DATASET_DIR = "/kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset"

assert os.path.exists(DATASET_DIR), DATASET_DIR

print("Dataset:", DATASET_DIR)

Dataset: /kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset


In [15]:
train_images = None
train_labels = None
val_images = None
val_labels = None

for root, dirs, files in os.walk(DATASET_DIR):

    path = root.replace("\\", "/")

    if path.endswith("/train/images"):
        train_images = root

    elif path.endswith("/train/labels"):
        train_labels = root

    elif path.endswith("/valid/images"):
        val_images = root

    elif path.endswith("/valid/labels"):
        val_labels = root

print("Train images:", train_images)
print("Train labels:", train_labels)
print("Validation images:", val_images)
print("Validation labels:", val_labels)

Train images: /kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset/train/images
Train labels: /kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset/train/labels
Validation images: /kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset/valid/images
Validation labels: /kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset/valid/labels


In [16]:
from pathlib import Path

train_image_files = [
    p for p in Path(train_images).iterdir()
    if p.is_file()
]

train_label_files = [
    p for p in Path(train_labels).iterdir()
    if p.is_file()
]

val_image_files = [
    p for p in Path(val_images).iterdir()
    if p.is_file()
]

val_label_files = [
    p for p in Path(val_labels).iterdir()
    if p.is_file()
]

print("Training images:", len(train_image_files))
print("Training labels:", len(train_label_files))

print("Validation images:", len(val_image_files))
print("Validation labels:", len(val_label_files))

Training images: 1012
Training labels: 1012
Validation images: 347
Validation labels: 348


In [17]:
import yaml
import os

data_config = {
    "path": DATASET_DIR,

    "train": os.path.relpath(
        train_images,
        DATASET_DIR
    ),

    "val": os.path.relpath(
        val_images,
        DATASET_DIR
    ),

    "nc": 1,

    "names": [
        "drone"
    ]
}

KAGGLE_YAML = "/kaggle/working/drone.yaml"

with open(KAGGLE_YAML, "w") as f:
    yaml.safe_dump(
        data_config,
        f,
        sort_keys=False
    )

print(open(KAGGLE_YAML).read())

path: /kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset
train: train/images
val: valid/images
nc: 1
names:
- drone



In [8]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

print("Model loaded successfully.")

Model loaded successfully.


In [10]:
results = model.val(
    data=KAGGLE_YAML,
    imgsz=320,
    batch=16,
    device=0
)

Ultralytics 8.4.128 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.1±0.3 ms, read: 22.2±25.5 MB/s, size: 132.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset/valid/labels... 347 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 347/347 232.3it/s 1.5s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 5.2it/s 4.2s0.1ss
                   all        347        369     0.0309      0.322     0.0129    0.00474
                person        347       

In [11]:
from ultralytics import YOLO
import torch

model = YOLO("yolo11n.pt")

device = 0 if torch.cuda.is_available() else "cpu"

results = model.train(

    # Dataset
    data=KAGGLE_YAML,

    # Model input
    imgsz=320,

    # Training
    epochs=100,
    batch=16,

    # GPU
    device=device,

    # Data loading
    workers=4,

    # Pretrained weights
    pretrained=True,

    # Early stopping
    patience=20,

    # Reproducibility
    seed=42,

    # Save results
    project="/kaggle/working/runs",
    name="yolo11n_drone_320",

    # Save checkpoints
    save=True,
    save_period=10,

    # Don't cache entire dataset in RAM
    cache=False,

    # Mixed precision training on GPU
    amp=True,

    verbose=True
)

Ultralytics 8.4.128 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/drone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_drone_320, nbs=64, nms=Fa

In [20]:
from pathlib import Path

RUN_DIR = Path(
    "/kaggle/working/runs/yolo11n_drone_320"
)

BEST_PT = (
    RUN_DIR /
    "weights" /
    "best.pt"
)

LAST_PT = (
    RUN_DIR /
    "weights" /
    "last.pt"
)

print("Best:", BEST_PT)
print("Last:", LAST_PT)

assert BEST_PT.exists()

Best: /kaggle/working/runs/yolo11n_drone_320/weights/best.pt
Last: /kaggle/working/runs/yolo11n_drone_320/weights/last.pt


In [18]:
trained_model = YOLO(
    str(BEST_PT)
)

metrics = trained_model.val(
    data=KAGGLE_YAML,
    imgsz=320,
    batch=16,
    device=0
)

Ultralytics 8.4.128 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 12.1±12.0 MB/s, size: 224.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset/valid/labels... 347 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 347/347 153.7it/s 2.3s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/muki2003/yolo-drone-detection-dataset/drone_dataset/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 4.9it/s 4.4s0.1s
                   all        347        369      0.896      0.816      0.896      0.539
Speed: 0.3ms preprocess, 2.7ms inference,

In [14]:
print(
    "Precision:",
    float(metrics.box.mp)
)

print(
    "Recall:",
    float(metrics.box.mr)
)

print(
    "mAP50:",
    float(metrics.box.map50)
)

print(
    "mAP50-95:",
    float(metrics.box.map)
)

Precision: 0.8959475037280726
Recall: 0.8157181571815718
mAP50: 0.895790544699462
mAP50-95: 0.5386730526525486


In [15]:
import glob

image_files = []

for ext in [
    "*.jpg",
    "*.jpeg",
    "*.png",
    "*.JPG",
    "*.JPEG",
    "*.PNG"
]:

    image_files.extend(
        glob.glob(
            os.path.join(
                val_images,
                ext
            )
        )
    )

print(
    "Validation images:",
    len(image_files)
)

Validation images: 347


In [16]:
prediction_results = trained_model.predict(

    source=image_files[:20],

    imgsz=320,

    conf=0.25,

    device=0,

    save=True,

    project="/kaggle/working/predictions",

    name="yolo11n_drone"
)


0: 320x320 1 drone, 1.4ms
1: 320x320 1 drone, 1.4ms
2: 320x320 1 drone, 1.4ms
3: 320x320 1 drone, 1.4ms
4: 320x320 1 drone, 1.4ms
5: 320x320 1 drone, 1.4ms
6: 320x320 1 drone, 1.4ms
7: 320x320 1 drone, 1.4ms
8: 320x320 (no detections), 1.4ms
9: 320x320 (no detections), 1.4ms
10: 320x320 1 drone, 1.4ms
11: 320x320 1 drone, 1.4ms
12: 320x320 1 drone, 1.4ms
13: 320x320 1 drone, 1.4ms
14: 320x320 (no detections), 1.4ms
15: 320x320 4 drones, 1.4ms
16: 320x320 1 drone, 1.4ms
17: 320x320 2 drones, 1.4ms
18: 320x320 1 drone, 1.4ms
19: 320x320 1 drone, 1.4ms
Speed: 0.7ms preprocess, 1.4ms inference, 0.6ms postprocess per image at shape (1, 3, 320, 320)
Results saved to /kaggle/working/predictions/yolo11n_drone


In [17]:
ncnn_export = trained_model.export(

    format="ncnn",

    imgsz=320,

    batch=1,

    device="cpu"
)

print(
    "NCNN export:",
    ncnn_export
)

Ultralytics 8.4.128 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ NCNN export does not support end2end models, disabling end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino

PyTorch: starting from '/kaggle/working/runs/yolo11n_drone_320/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 5, 2100) (5.2 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 185ms
Prepared 1 package in 125ms
Installed 1 package in 5ms
 + ncnn==1.0.20260526

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirement ['pnnx==20260526'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 29 packages in 177ms
Prepar

pnnxparam = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,320,320]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,320,320]f32
############# pass_level0
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.modules.block.Bottleneck
inlin

NCNN: export success ✅ 4.9s, saved as '/kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model' (10.0 MB)

Export complete (5.1s)
Results saved to /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model imgsz=320 
Validate:        yolo val task=detect model=/kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model imgsz=320 data=/kaggle/working/drone.yaml  
Visualize:       https://netron.app
NCNN export: /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model


In [18]:
from pathlib import Path

NCNN_DIR = Path(
    str(ncnn_export)
)

print("NCNN directory:")
print(NCNN_DIR)

print("\nFiles:")

for file in NCNN_DIR.rglob("*"):

    if file.is_file():

        print(
            file.name,
            "-",
            round(
                file.stat().st_size / (1024 * 1024),
                2
            ),
            "MB"
        )

NCNN directory:
/kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model

Files:
model_ncnn.py - 0.0 MB
model.ncnn.bin - 9.88 MB
model.ncnn.param - 0.02 MB
metadata.yaml - 0.0 MB
model_pnnx.cpython-312.pyc - 0.06 MB


In [19]:
try:

    fp16_export = trained_model.export(

        format="ncnn",

        imgsz=320,

        batch=1,

        quantize=16,

        device="cpu"
    )

    print(
        "FP16 NCNN:",
        fp16_export
    )

except Exception as e:

    print(
        "FP16 export failed:"
    )

    print(e)

Ultralytics 8.4.128 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ NCNN export does not support end2end models, disabling end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino

PyTorch: starting from '/kaggle/working/runs/yolo11n_drone_320/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 5, 2100) (5.2 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 231ms
Prepared 1 package in 215ms
Installed 1 package in 5ms
 + ncnn==1.0.20260526

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirement ['pnnx==20260526'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 29 packages in 329ms
Prepar

pnnxparam = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model/model_ncnn.py
fp16 = 1
optlevel = 2
device = cpu
inputshape = [1,3,320,320]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,320,320]f32
############# pass_level0
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.modules.block.Bottleneck
inlin

NCNN: export success ✅ 5.7s, saved as '/kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model' (5.1 MB)

Export complete (5.9s)
Results saved to /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model imgsz=320 quantize=16
Validate:        yolo val task=detect model=/kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model imgsz=320 data=/kaggle/working/drone.yaml quantize=16 
Visualize:       https://netron.app
FP16 NCNN: /kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model


In [21]:
from pathlib import Path

NCNN_DIR = Path(
    str(fp16_export)
)

print("NCNN directory:")
print(NCNN_DIR)

print("\nFiles:")

for file in NCNN_DIR.rglob("*"):

    if file.is_file():

        print(
            file.name,
            "-",
            round(
                file.stat().st_size / (1024 * 1024),
                2
            ),
            "MB"
        )

NCNN directory:
/kaggle/working/runs/yolo11n_drone_320/weights/best_ncnn_model

Files:
model.ncnn.param - 0.02 MB
metadata.yaml - 0.0 MB
model_ncnn.py - 0.0 MB
model.ncnn.bin - 4.96 MB
model_pnnx.cpython-312.pyc - 0.06 MB
